# Tutorial: WorldQuant Brain — Bigdata.com Co-mentions API

Discover which companies, places, and concepts are talked about together with your topic or a focal company—then turn that into network views or thematic baskets. This tutorial shows how to use the Co-mentions API to discover **entities that are frequently mentioned together** with your search query. Co-mentions help you see which companies, places, people, and concepts appear alongside a topic or focal entity.

**What you'll learn**
- Run a co-mentions query with a text query and optional time and entity filters
- Interpret the response (entities by category: companies, places, organizations, people, products, concepts)
- Visualize co-mention relationships as network graphs centered on a focal entity
- Build a thematic basket of top companies around a topic (e.g. cloud computing) and plot the results

**Why Co-mentions**
- Discover who or what is talked about together with your topic—useful for competitive mapping, supply-chain and theme discovery, and event context
- Same semantic search and entity filters as the Search API, so you can narrow by time window and by entity (e.g. only documents mentioning Apple)
- Built for research and backtesting (e.g. point-in-time, configurable time windows)

**Example use cases:** competitive and theme mapping, supply-chain and partnership discovery, event and narrative context around a company or topic, thematic basket construction.

**Signals:** Co-mentions help define thematic baskets and entity context; the same entity IDs feed into search filters and signal construction in Workflow_example.

**In this notebook**
1. **Co-mentions API** — run a basic co-mentions query (topic only), interpret the response by category (companies, places, people, etc.), and understand the response structure.
2. **Co-mentions network graph (exploration)** — add an entity filter and visualize co-mentioned entities as network graphs by category.
3. **Thematic basket (actionable universe)** — build two baskets by headline/chunk count, resolve IDs to names, plot both sets; note on lookahead at the end.

**Prerequisites:** `.env` file with `BRAIN_EMAIL` and `BRAIN_PASSWORD` (see `.env.example`). For access, see the hackathon or [WorldQuant Brain](https://www.worldquantbrain.com/) documentation.

**Time:** about 15–20 minutes.

**Endpoint documentation:** [Co-mentions API reference](https://docs.bigdata.com/api-reference/co-mentions/connected-entities)

In [ ]:
from session import BrainSession
from print_helpers import print_comention_results

## Setup & Configuration

In [ ]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# WorldQuant Brain API Configuration
API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")

# API Endpoints
SEARCH_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/search"
COMENTIONS_ENDPOINT = f"{SEARCH_ENDPOINT}/co-mentions/entities"
KG_ENTITIES_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/knowledge-graph/entities/id"

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

## Key considerations before starting

The provider APIs do not have full coverage for parameter validation—double-check your request against the [API reference](https://docs.bigdata.com/api-reference) to avoid issues from typos or invalid values. For historical windows (e.g. 2021–2022), set `freshness_boost` to `0` so results are not biased toward recent data.

In [ ]:
# Date range for examples
START_DATE = "2021-01-01"
END_DATE = "2021-12-30"
TEXT = "cloud computing market growth"
ENTITY_ID = "D8442A"  # Apple Inc

## 1. Co-mentions API

The Co-mentions API returns **entities** that are frequently **co-mentioned** with your search query (by category: e.g. companies, places, people—see 1.1 for the full list). It uses the same semantic search as the Search API (meaning-based, not just keywords). You can filter by timestamp and, optionally, by entity.

In the example below, we find entities (companies, places, products, etc.) that are most **co-mentioned with the topic**—e.g. "cloud computing market growth"—over the given time window. This is a **topic-only** call: no entity filter. In the **next section** we add an **entity** filter (e.g. Apple) to find companies co-mentioned **with that company** in the context of the same topic, and we plot them as network graphs.

Before we run the query, two parameters are worth setting explicitly.

`auto_enrich_filters`
- When set to `True`, entities are automatically extracted from the text using NLP and added to the `any_of` filter by default. This can introduce unintended results, especially with common entities like places.
- When set to `False` (**recommended**), you retain full control over entity-based filtering.


In [ ]:
comention_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            }
        },
        "ranking_params": {"freshness_boost": 0}
    },
    "limit": 10
}

response = session.post(COMENTIONS_ENDPOINT, json=comention_query)
comentions_data = response.json()
print_comention_results(response, comentions_data)

You'll see entities grouped by category—companies, places, products, concepts—with chunk and headline counts. Try changing the query text or date range and re-running the co-mentions call above to see how results change.

### 1.1 Response structure

Understanding what the API returns.

**How the endpoint builds the result:** The API returns top N by chunk volume and top N by headline volume, then **merges** the two lists. In the merged result, an entity can have both counts, only chunk count, or only headline count.

The list is **top entities overall** (all types), not only companies—so you get companies, places, people, products, organizations, and concepts. To get "top companies," you use `results.companies` and rank by the metric you care about (headlines or chunks). The `limit` parameter caps how many entities are returned (see [API reference](https://docs.bigdata.com/api-reference/search/get-co-mentions)); use a higher value (e.g. 500) when building thematic baskets.

In short: you get a merged, categorized list of entities, each with chunk and/or headline counts.

**Response contains three main sections:**

**1. `results`:** Entities grouped by category (only categories that have entities in the merged top-N list)
- `companies`: Companies (e.g. suppliers, competitors)
- `places`: Geographic entities
- `organizations`: Non-company organizations
- `people`: Named persons
- `products`: Products and brands
- `concepts`: Topics and themes

Each entity includes an identifier (for the knowledge graph), `total_chunks_count`, and/or `total_headlines_count` (one or both, depending on whether it came from the chunk list, headline list, or both).

**2. `metadata`:** Request information (e.g. `request_id`, `timestamp`).

**3. `usage`:** API consumption metrics (e.g. `api_query_units`).

## 2. Co-mentions network graph (exploration)

Next, we narrow the view to a **single company**: only documents that mention that company are used, and we visualize who or what is co-mentioned with it. Here we add an **entity** filter (e.g. Apple) so that only documents mentioning that company are considered. We then find entities that are **co-mentioned with that company** in the context of the topic (e.g. "cloud computing market growth") and plot them by category as network graphs. The focal entity sits at the center of the graph—useful for competitive sets, geographic clusters, or key people and concepts tied to a topic and a company.

In [ ]:
from api_helpers import create_comentions_network_graph

# Query co-mentions filtered by entity
comention_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {"search_in": "ALL", "any_of": [ENTITY_ID]}
        },
        "ranking_params": {
            "freshness_boost": 0
        }
    },
    "limit": 100
}

response = session.post(COMENTIONS_ENDPOINT, json=comention_query)
comentions_data = response.json()

# Display network graphs
if comentions_data.get("results"):
    for category in ["companies", "places", "organizations", "people", "products", "concepts"]:
        entities = comentions_data["results"].get(category, [])
        if entities:
            fig = create_comentions_network_graph(
                session=session,
                kg_entities_endpoint=KG_ENTITIES_ENDPOINT,
                center_name="Apple Inc.",
                center_id=ENTITY_ID,
                connected_entities=entities,
                category_name=category,
                text=TEXT,
                max_nodes=15
            )
            if fig:
                fig.show()

Try a different focal entity (e.g. another company ID) or topic to explore other networks.

## 3. Thematic basket 

We now switch to building a **thematic basket**—a list of top companies around a topic—using a topic-only query. As in 1.1, the API returns a merged list (top-by-chunks and top-by-headlines); we take `results.companies` and build two rankings—**top 10 by headline count** and **top 10 by chunk count**—then resolve IDs via the Knowledge Graph and plot both. **You** can use these entity IDs in the Search or Volume API (e.g. `entity.any_of`) for filtering.

The approach here is **very simple** and based only on **volume** (chunk and headline counts). A proper approach would post-process the chunks (e.g. via the Search API) to verify and enrich the relationships.

This section uses a different topic and date variables (`BASKET_TOPIC`, `BASKET_START`, `BASKET_END`) so you can run it independently. **Example:** "cloud computing market growth" — we get companies from the co-mentions result, build two sets (top 10 by headlines, top 10 by chunks), resolve IDs to names, then display both baskets and plot them. See the note on **lookahead and rolling baskets** at the end of this section before using baskets in backtests.

In [ ]:
# Thematic basket: "cloud computing market growth" — two sets (by headlines, by chunks)
import pandas as pd

BASKET_TOPIC = "cloud computing market growth"
BASKET_START = "2021-01-01"
BASKET_END = "2021-12-30"
TOP_N_COMPANIES = 10

# Co-mentions query (topic only, no entity filter)
basket_query = {
    "query": {
        "text": BASKET_TOPIC,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{BASKET_START}T00:00:00Z",
                "end": f"{BASKET_END}T23:59:59Z"
            }
        },
        "ranking_params": {"freshness_boost": 0},
        "limit": 500
    },
}

response_basket = session.post(COMENTIONS_ENDPOINT, json=basket_query)
basket_data = response_basket.json()

# API returns merged top-by-chunks and top-by-headlines; we take companies only.
companies_raw = basket_data.get("results", {}).get("companies", [])

# Two baskets: top N companies BY HEADLINE count, top N companies BY CHUNK count.
# Filter to entities that have non-zero count for that metric (merged list can have
# only-headline or only-chunk entities), then take top N. Larger limit helps get more
# companies with both metrics.
with_headlines = [c for c in companies_raw if c.get("total_headlines_count", 0) > 0]
with_chunks = [c for c in companies_raw if c.get("total_chunks_count", 0) > 0]
top_by_headlines = sorted(
    with_headlines,
    key=lambda x: x.get("total_headlines_count", 0),
    reverse=True
)[:TOP_N_COMPANIES]
top_by_chunks = sorted(
    with_chunks,
    key=lambda x: x.get("total_chunks_count", 0),
    reverse=True
)[:TOP_N_COMPANIES]

all_ids = list({e["id"] for e in top_by_headlines + top_by_chunks})
kg_response = session.post(KG_ENTITIES_ENDPOINT, json={"values": all_ids})
resolved = kg_response.json().get("results", {}) if kg_response.status_code == 200 else {}

def make_basket_df(entities):
    rows = []
    for i, ent in enumerate(entities, 1):
        name = resolved.get(ent["id"], {}).get("name", ent["id"])
        rows.append({
            "rank": i,
            "entity_id": ent["id"],
            "name": name,
            "headlines": ent.get("total_headlines_count", 0),
            "chunks": ent.get("total_chunks_count", 0),
        })
    return pd.DataFrame(rows)

if not companies_raw:
    print("No companies found for this topic.")
    basket_by_headlines_df = pd.DataFrame()
    basket_by_chunks_df = pd.DataFrame()
else:
    basket_by_headlines_df = make_basket_df(top_by_headlines)
    basket_by_chunks_df = make_basket_df(top_by_chunks)
    print(f"Thematic basket: top {TOP_N_COMPANIES} companies by headlines and top {TOP_N_COMPANIES} by chunks | \"{BASKET_TOPIC}\" ({BASKET_START} to {BASKET_END})")
    if len(with_chunks) < TOP_N_COMPANIES:
        print(f"Note: the API returned only {len(with_chunks)} companies with non-zero chunk counts for this topic/period; the by-chunks basket has {len(with_chunks)} rows.")
    print("\n--- Top 10 companies by headline count ---")
    display(basket_by_headlines_df)
    print("\n--- Top 10 companies by chunk count ---")
    display(basket_by_chunks_df)

In [ ]:
# Graphics: bar charts for both baskets (top 10 by headlines, top 10 by chunks)
import plotly.express as px

try:
    has_headlines = basket_by_headlines_df is not None and not basket_by_headlines_df.empty
    has_chunks = basket_by_chunks_df is not None and not basket_by_chunks_df.empty
except NameError:
    has_headlines = has_chunks = False

if has_headlines:
    fig_headlines = px.bar(
        basket_by_headlines_df.sort_values("headlines", ascending=True),
        x="headlines",
        y="name",
        orientation="h",
        title=f'Top 10 companies by headline count<br><sup>"{BASKET_TOPIC}" | {BASKET_START} to {BASKET_END}</sup>',
        labels={"headlines": "Headline count", "name": "Company"},
        text="headlines",
        text_auto=",.0f",
    )
    fig_headlines.update_layout(
        height=350 + len(basket_by_headlines_df) * 22,
        margin=dict(l=20),
        yaxis=dict(autorange="reversed", tickfont=dict(size=10)),
        showlegend=False,
    )
    fig_headlines.update_traces(textposition="outside")
    fig_headlines.show()

if has_chunks:
    fig_chunks = px.bar(
        basket_by_chunks_df.sort_values("chunks", ascending=True),
        x="chunks",
        y="name",
        orientation="h",
        title=f'Top 10 companies by chunk count<br><sup>"{BASKET_TOPIC}" | {BASKET_START} to {BASKET_END}</sup>',
        labels={"chunks": "Chunk count", "name": "Company"},
        text="chunks",
        text_auto=",.0f",
    )
    fig_chunks.update_layout(
        height=350 + len(basket_by_chunks_df) * 22,
        margin=dict(l=20),
        yaxis=dict(autorange="reversed", tickfont=dict(size=10)),
        showlegend=False,
    )
    fig_chunks.update_traces(textposition="outside")
    fig_chunks.show()

if not has_headlines and not has_chunks:
    print("Run the cell above first to build the baskets.")

### 3.1 Lookahead and thematic baskets in backtests

Why this matters for backtests:

When you build a thematic basket using a **single time window** (e.g. one year of data), the resulting company list is **in-sample** for that period: you are selecting companies based on information from that same period. Using that basket as if it were known at the start of the window in a backtest would introduce **lookahead bias**, because in reality you would not have known the “top comentioned” list at the beginning of the period.

For **real backtests**, use a **rolling (expanding or rolling-window) approach**: e.g. at each evaluation date, build the basket using only data **up to that date** (or a lagged window ending before that date), then apply the basket in the next period. That way the basket is formed without using future information and avoids lookahead.